# V2 Transformer Training (Patched Audio)

Trains a **Transformer Fusion V2** model on V2-patched features (`train_features_v2_patched.pt`).

**Key improvements over V1:**
- 5 tokens: `[CLS] [F0] [F1] [F2] [AUD]` — 3 image frames let the transformer learn temporal attention
- Real VGGish audio (patched) — ~88% coverage; `z_aud` L2-normalised per sample

**This revision (after the over-regularized "opt" run underfit):**
- **`z_img` L2-normalised per frame** — the key fix; nearly doubled MLP retrieval, so kept
- **Silent audio masked in attention** via `src_key_padding_mask` (correctness)
- **LR warmup → cosine**, 80 epochs (kept — stability + the val curve was still improving)
- **Regularization dialed back**: dropout 0.1 (was 0.2), modality dropout 0.05 (was 0.15), weight_decay 1e-4 (was 0.05), removed the separate input-dropout. The opt run stacked too much reg → train loss stuck at 0.91 (underfit) and retrieval dropped.

## Step 1: Imports & Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 2: Dataset

In [ ]:
class MultimodalDatasetV2(Dataset):
    """
    Loads V2-patched feature files.
    z_img frames AND z_aud are both L2-normalised per sample, so the two
    modalities enter the transformer on the same (unit) scale.
    Silent clips keep an all-zero audio vector and are masked out in attention.
    """
    def __init__(self, file_path):
        data = torch.load(file_path, map_location="cpu", weights_only=False)
        self.z_img     = data["z_img"].float()      # [N, 3, 512]
        self.z_aud     = data["z_aud"].float()      # [N, 128]
        self.v_teacher = data["v_teacher"].float()  # [N, 1024]
        self.has_audio = data.get("has_audio", torch.ones(len(self.z_img), dtype=torch.bool))
        assert len(self.z_img) == len(self.z_aud) == len(self.v_teacher)

        # --- Input normalisation (the key fix) ---
        # CLIP embeddings sit on a sphere of radius ~10.6 with per-sample
        # magnitude variance, and CLIP itself is trained with cosine similarity.
        # L2-normalise every frame: kills the magnitude nuisance signal and puts
        # images on the same unit scale as the audio token.
        self.z_img = F.normalize(self.z_img, dim=-1)

        # VGGish audio is raw log-mel (L2~1456). Per-sample L2-norm -> unit scale.
        # Silent clips (has_audio=False) are all-zero and stay exactly zero.
        aud_norms = self.z_aud.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        self.z_aud = self.z_aud / aud_norms
        self.z_aud[~self.has_audio] = 0.0

        n_aud = self.has_audio.sum().item()
        print(f"Loaded {len(self.z_img)} samples | z_img={tuple(self.z_img.shape)}")
        print(f"  z_img L2 (post-norm): {self.z_img.norm(dim=-1).mean().item():.3f}")
        print(f"  Audio: {n_aud}/{len(self.z_img)} ({100*n_aud/len(self.z_img):.1f}%)")

    def __len__(self):
        return len(self.z_img)

    def __getitem__(self, idx):
        return {
            "z_img":     self.z_img[idx],
            "z_aud":     self.z_aud[idx],
            "v_teacher": self.v_teacher[idx],
            "has_audio": self.has_audio[idx],  # bool scalar
        }

## Step 3: Load Data

In [ ]:
def find_features(filename):
    """Locate a feature file locally or inside a Kaggle input dataset."""
    candidates = [
        filename,
        os.path.join("data_V2_patched", filename),
        os.path.join("..", "data_V2_patched", filename),
        os.path.join("..", "features", "patched_features", filename),
        os.path.join("/kaggle/working", filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    if os.path.exists("/kaggle/input"):
        import glob
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"{filename} not found.\n"
        "  On Kaggle : attach the msrvtt-v2-patched dataset as an input.\n"
        "  Locally   : place files in data_V2_patched/."
    )

# On Kaggle, write models to /kaggle/working so they appear as output files
SAVE_DIR = "/kaggle/working/models_v2" if os.path.exists("/kaggle/working") else "models_v2"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model save directory: {SAVE_DIR}")

train_ds = MultimodalDatasetV2(find_features("train_features_v2_patched.pt"))
test_ds  = MultimodalDatasetV2(find_features("test_features_v2_patched.pt"))

val_size   = int(0.10 * len(train_ds))
train_size = len(train_ds) - val_size
train_split, val_split = random_split(train_ds, [train_size, val_size],
                                      generator=torch.Generator().manual_seed(42))

# Batch 256: more in-batch negatives -> stronger InfoNCE signal (the MLP showed
# the contrastive term is starved at 128).
train_loader = DataLoader(train_split, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_split,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,     batch_size=256, shuffle=False, num_workers=2)

print(f"Split: train={train_size}, val={val_size}, test={len(test_ds)}")

## Step 4: Transformer Fusion V2

```
Tokens (5 total):  [CLS] [FRAME_0] [FRAME_1] [FRAME_2] [AUDIO]
Type IDs:            0       1         1         1         2
Position IDs:        0       1         2         3         4
```

Each frame token is an independent CLIP embedding projected to `embed_dim`. The audio token is the VGGish embedding projected to `embed_dim`. The CLS token reads all 4 and produces the final embedding.

In [ ]:
class TransformerFusionV2(nn.Module):
    """
    5-token Transformer: [CLS] [F0] [F1] [F2] [AUD].
    Modality type embeddings: 0=CLS, 1=IMG_FRAME, 2=AUDIO.
    Positional embeddings distinguish the 3 frame + audio positions.

    Silent / dropped audio is handled with a real attention padding mask, so the
    audio token is fully excluded from attention (its modality+pos embeddings no
    longer leak into the representation, which the old value-zeroing left behind).

    NOTE: the first "opt" run stacked dropout 0.2 + input-dropout + modality-drop
    0.15 + wd 0.05, which pushed this small model into *underfitting* (train loss
    stuck at ~0.91, retrieval dropped). Regularization is now light: dropout 0.1,
    light modality dropout, no separate input dropout.
    """
    def __init__(self, img_dim=512, aud_dim=128, embed_dim=384,
                 num_heads=6, num_layers=3, output_dim=1024,
                 dropout=0.1, num_frames=3, aud_drop_prob=0.05):
        super().__init__()
        self.num_frames = num_frames
        self.num_tokens = 1 + num_frames + 1  # CLS + frames + audio = 5
        self.aud_drop_prob = aud_drop_prob    # modality dropout (train only)

        self.img_proj = nn.Linear(img_dim, embed_dim)
        self.aud_proj = nn.Linear(aud_dim, embed_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.modality_embed = nn.Embedding(3, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,  # Pre-LN for stable training
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, output_dim),
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, z_img, z_aud, has_audio=None):
        """
        z_img     : [B, 3, 512]  — 3 CLIP frame embeddings (L2-normalised)
        z_aud     : [B, 128]     — VGGish audio, L2-normalised; zero for silent clips
        has_audio : [B] bool     — True where real audio is present
        Returns   : [B, 1024]
        """
        B = z_img.size(0)
        device = z_img.device
        if has_audio is None:
            has_audio = torch.ones(B, dtype=torch.bool, device=device)
        else:
            has_audio = has_audio.to(device).bool()

        # Modality dropout (light): in training, occasionally hide the audio token
        # even when present, so the CLS head stays robust to the ~12% of clips
        # that are genuinely silent at test time.
        audio_keep = has_audio
        if self.training and self.aud_drop_prob > 0:
            rand_keep = torch.rand(B, device=device) >= self.aud_drop_prob
            audio_keep = has_audio & rand_keep

        img_tokens = self.img_proj(z_img)               # [B, 3, embed_dim]
        aud_token  = self.aud_proj(z_aud).unsqueeze(1)  # [B, 1, embed_dim]
        cls_token  = self.cls_token.expand(B, -1, -1)   # [B, 1, embed_dim]

        # Concatenate: [CLS, F0, F1, F2, AUD]
        tokens = torch.cat([cls_token, img_tokens, aud_token], dim=1)  # [B, 5, E]

        type_ids = torch.tensor(
            [0] + [1] * self.num_frames + [2],
            dtype=torch.long, device=device,
        )
        tokens = tokens + self.modality_embed(type_ids) + self.pos_embed

        # Padding mask: True = ignore this key in attention. CLS + 3 frames are
        # always present; the audio token is masked for silent / dropped clips.
        # No row is ever fully masked, so there is no NaN risk.
        pad_mask = torch.zeros(B, self.num_tokens, dtype=torch.bool, device=device)
        pad_mask[:, -1] = ~audio_keep

        features = self.transformer(tokens, src_key_padding_mask=pad_mask)  # [B, 5, E]
        cls_rep  = features[:, 0]             # [B, embed_dim]
        return self.head(cls_rep)             # [B, 1024]


model = TransformerFusionV2().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"TransformerFusionV2 parameters: {n_params:,}")

# Sanity-check forward pass with has_audio mask
with torch.no_grad():
    dummy_img       = torch.randn(4, 3, 512).to(device)
    dummy_aud       = torch.randn(4, 128).to(device)
    dummy_has_audio = torch.tensor([True, True, False, True])
    out = model(dummy_img, dummy_aud, dummy_has_audio)
    print(f"Output shape: {out.shape}  (expected [4, 1024])")

## Step 5: Loss — Joint Cosine + InfoNCE

Same strategy that gave MedR=1 in V1. The `alpha` parameter controls the blend:
- Cosine term: absolute alignment (points in the right direction)
- InfoNCE term: relative separation (discriminates between videos)

In [ ]:
class JointLoss(nn.Module):
    def __init__(self, temperature=0.07, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, pred, target):
        pred_n   = F.normalize(pred,   dim=-1)
        target_n = F.normalize(target, dim=-1)

        # Cosine loss
        cos_loss = (1 - (pred_n * target_n).sum(-1)).mean()

        # Symmetric InfoNCE
        N = pred_n.size(0)
        logits   = torch.matmul(pred_n, target_n.T) / self.temperature
        labels   = torch.arange(N, device=pred.device)
        nce_loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

        return (1 - self.alpha) * cos_loss + self.alpha * nce_loss


criterion = JointLoss(temperature=0.07, alpha=0.5)
print("JointLoss defined (50% cosine + 50% InfoNCE).")

## Step 6: Train

In [ ]:
EPOCHS = 80
WARMUP = 5
os.makedirs(SAVE_DIR, exist_ok=True)

# Light weight decay (1e-4). The opt run used 0.05 which, stacked with heavy
# dropout, caused underfitting. Keep regularization gentle and let normalization
# + the longer schedule do the work.
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

# Linear warmup (5 ep) -> cosine decay. Warmup stabilises the early transformer
# updates that plain cosine-from-cold can derail.
warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP)
cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP, eta_min=1e-5)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP]
)

train_losses, val_losses = [], []
best_val = float("inf")

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss, total_n = 0.0, 0
    for batch in train_loader:
        z_img     = batch["z_img"].to(device)
        z_aud     = batch["z_aud"].to(device)
        v_teacher = batch["v_teacher"].to(device)
        has_audio = batch["has_audio"].to(device)
        optimizer.zero_grad()
        pred = model(z_img, z_aud, has_audio)
        loss = criterion(pred, v_teacher)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * z_img.size(0)
        total_n    += z_img.size(0)
    train_loss = total_loss / total_n

    # --- Val ---
    model.eval()
    val_loss, val_n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            z_img     = batch["z_img"].to(device)
            z_aud     = batch["z_aud"].to(device)
            v_teacher = batch["v_teacher"].to(device)
            has_audio = batch["has_audio"].to(device)
            pred = model(z_img, z_aud, has_audio)
            loss = criterion(pred, v_teacher)
            val_loss += loss.item() * z_img.size(0)
            val_n    += z_img.size(0)
    val_loss /= val_n
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "transformer_v2_best.pt"))

    if (epoch + 1) % 5 == 0 or epoch == 0:
        marker = "  *" if val_loss == best_val else ""
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}{marker}")
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"transformer_v2_ep{epoch+1}.pt"))

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "transformer_v2_final.pt"))
print(f"\nDone. Best val loss: {best_val:.4f}")

## Step 7: Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label="Train", linewidth=1.5)
ax.plot(val_losses,   label="Val",   linewidth=1.5, linestyle="--")
ax.set_title("Transformer V2 — Joint Loss (Cosine + InfoNCE)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "transformer_v2_loss.png"), dpi=150)
plt.show()
print(f"Saved loss curve to {SAVE_DIR}/transformer_v2_loss.png")

## Step 8: Quick Test-Set Retrieval Metrics

In [ ]:
model.load_state_dict(torch.load(
    os.path.join(SAVE_DIR, "transformer_v2_best.pt"),
    map_location=device, weights_only=True,
))
model.eval()

preds, targets = [], []
with torch.no_grad():
    for batch in test_loader:
        z_img     = batch["z_img"].to(device)
        z_aud     = batch["z_aud"].to(device)
        has_audio = batch["has_audio"].to(device)
        pred = model(z_img, z_aud, has_audio)
        preds.append(pred.cpu())
        targets.append(batch["v_teacher"])

preds   = torch.cat(preds)
targets = torch.cat(targets)

pred_n   = F.normalize(preds,   dim=-1)
target_n = F.normalize(targets, dim=-1)

cos_sim = (pred_n * target_n).sum(-1).mean().item()
mse     = F.mse_loss(preds, targets).item()

sim_mat = torch.matmul(pred_n, target_n.T)
N = sim_mat.size(0)
ranks = [(sim_mat[i] > sim_mat[i, i]).sum().item() + 1 for i in range(N)]
ranks = torch.tensor(ranks, dtype=torch.float)

print("=== Transformer Fusion V2 — Patched Audio (Test Set) ===")
print(f"  Cosine Sim : {cos_sim:.4f}")
print(f"  MSE        : {mse:.4f}")
print(f"  R@1  : {(ranks<=1).float().mean().item()*100:.1f}%")
print(f"  R@5  : {(ranks<=5).float().mean().item()*100:.1f}%")
print(f"  R@10 : {(ranks<=10).float().mean().item()*100:.1f}%")
print(f"  MedR : {ranks.median().item():.0f}")